读入

In [39]:
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw/"
OUTPUT_DIR = "../data/processed/"

living_path = DATA_DIR + "生活水平.xlsx"

伪第一张表

In [40]:
df = pd.read_excel(
    living_path,
    sheet_name="人均可支配收入"
)

df.head()

,disposableIcome,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,城市名称,2015,2016,2017,2018,2019,2020,2021,2022
1,city3,40948,44632,48692,52936,57544,62520,64876,70842
2,city6,35140,38500,41940,45812,49628,54040,56244,61698
3,city14,13280,14548,16076,17716,19396,21052,22288,24624
4,city9,15264,16844,18400,19884,21348,23176,24600,26724


真第一张表


In [41]:
df = pd.read_excel(
    living_path,
    sheet_name="人均可支配收入",
    header=1
)

df.head()

,城市名称,2015,2016,2017,2018,2019,2020,2021,2022
0,city3,40948,44632,48692,52936,57544,62520,64876,70842
1,city6,35140,38500,41940,45812,49628,54040,56244,61698
2,city14,13280,14548,16076,17716,19396,21052,22288,24624
3,city9,15264,16844,18400,19884,21348,23176,24600,26724
4,city11,34316,37136,40116,43504,47380,51612,54664,60048


人均可支配收入转换长表

In [42]:
disposable = pd.read_excel(
    living_path,
    sheet_name="人均可支配收入",
    header=1
)

disposable_long = disposable.melt(
    id_vars=["城市名称"],
    var_name="year",
    value_name="disposable_income"
)

disposable_long = disposable_long.rename(
    columns={
        "城市名称": "city_id"
    }
)

disposable_long.head()

,city_id,year,disposable_income
0,city3,2015,40948
1,city6,2015,35140
2,city14,2015,13280
3,city9,2015,15264
4,city11,2015,34316


统一city_id year

In [43]:
disposable_long["city_id"] = (
    disposable_long["city_id"]
    .astype(str)
    .str.strip()
)

disposable_long["year"] = pd.to_numeric(
    disposable_long["year"],
    errors="coerce"
)

disposable_long.dtypes

city_id                str
year                 int64
disposable_income    int64
dtype: object

检查结果

In [44]:
print(disposable_long.shape)

print(
    disposable_long["city_id"].nunique()
)

print(
    disposable_long["year"].min(),
    disposable_long["year"].max()
)

print(
    disposable_long.duplicated(
        ["city_id", "year"]
    ).sum()
)

print(
    disposable_long.isna().sum()
)

(176, 3)
22
2015 2022
0
city_id              0
year                 0
disposable_income    0
dtype: int64


数据清洗过程改写成函数

In [45]:
def clean_living_sheet(
    file_path,
    sheet_name,
    value_name
):
    
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=1
    )

    df = df.melt(
        id_vars=["城市名称"],
        var_name="year",
        value_name=value_name
    )

    df = df.rename(
        columns={
            "城市名称": "city_id"
        }
    )

    df["city_id"] = (
        df["city_id"]
        .astype(str)
        .str.strip()
    )

    df["year"] = pd.to_numeric(
        df["year"],
        errors="coerce"
    )

    df = (
        df
        .sort_values(["city_id", "year"])
        .reset_index(drop=True)
    )

    return df

清洗6张表

In [46]:
disposable = clean_living_sheet(
    living_path,
    "人均可支配收入",
    "disposable_income"
)

consumption = clean_living_sheet(
    living_path,
    "人均消费支出",
    "consumption_expenditures"
)

urban_consumption = clean_living_sheet(
    living_path,
    "城镇居民消费支出",
    "urban_consumption_expenditures"
)

rural_consumption = clean_living_sheet(
    living_path,
    "农村居民消费支出",
    "rural_consumption_expenditures"
)

urban_income = clean_living_sheet(
    living_path,
    "城镇居民人均收入",
    "urban_disposable_income"
)

rural_income = clean_living_sheet(
    living_path,
    "农村居民人均收入",
    "rural_disposable_income"
)

质量检查

In [47]:
living_tables = {
    "人均可支配收入": disposable,
    "人均消费支出": consumption,
    "城镇居民消费支出": urban_consumption,
    "农村居民消费支出": rural_consumption,
    "城镇居民人均收入": urban_income,
    "农村居民人均收入": rural_income
}


for name, df in living_tables.items():

    print("=" * 60)
    print(name)

    print("Shape：", df.shape)
    print("城市数：", df["city_id"].nunique())

    print(
        "年份：",
        df["year"].min(),
        "-",
        df["year"].max()
    )

    print(
        "重复 city-year：",
        df.duplicated(
            ["city_id", "year"]
        ).sum()
    )

    print("缺失值：")
    print(df.isna().sum())

人均可支配收入
Shape： (176, 3)
城市数： 22
年份： 2015 - 2022
重复 city-year： 0
缺失值：
city_id              0
year                 0
disposable_income    0
dtype: int64
人均消费支出
Shape： (184, 3)
城市数： 23
年份： 2015 - 2022
重复 city-year： 0
缺失值：
city_id                     0
year                        0
consumption_expenditures    0
dtype: int64
城镇居民消费支出
Shape： (336, 3)
城市数： 28
年份： 2011 - 2022
重复 city-year： 0
缺失值：
city_id                            0
year                               0
urban_consumption_expenditures    39
dtype: int64
农村居民消费支出
Shape： (264, 3)
城市数： 22
年份： 2011 - 2022
重复 city-year： 0
缺失值：
city_id                            0
year                               0
rural_consumption_expenditures    40
dtype: int64
城镇居民人均收入
Shape： (396, 3)
城市数： 33
年份： 2011 - 2022
重复 city-year： 0
缺失值：
city_id                     0
year                        0
urban_disposable_income    35
dtype: int64
农村居民人均收入
Shape： (336, 3)
城市数： 28
年份： 2011 - 2022
重复 city-year： 0
缺失值：
city_id                     0
year             

合并六张表

In [48]:
living = pd.merge(
    disposable,
    consumption,
    on=["city_id", "year"],
    how="outer"
)

living = living.merge(
    urban_consumption,
    on=["city_id", "year"],
    how="outer"
)

living = living.merge(
    rural_consumption,
    on=["city_id", "year"],
    how="outer"
)

living = living.merge(
    urban_income,
    on=["city_id", "year"],
    how="outer"
)

living = living.merge(
    rural_income,
    on=["city_id", "year"],
    how="outer"
)

整理最终结果

In [49]:
living = (
    living
    .sort_values(
        ["city_id", "year"]
    )
    .reset_index(drop=True)
)

质量检查

In [50]:
print("Shape：", living.shape)

print(
    "城市数量：",
    living["city_id"].nunique()
)

print(
    "年份：",
    living["year"].min(),
    "-",
    living["year"].max()
)

print(
    "重复 city-year：",
    living.duplicated(
        ["city_id", "year"]
    ).sum()
)

print()

print("各列缺失值：")

print(
    living.isna().sum()
)

Shape： (432, 8)
城市数量： 36
年份： 2011 - 2022
重复 city-year： 0

各列缺失值：
city_id                             0
year                                0
disposable_income                 256
consumption_expenditures          248
urban_consumption_expenditures    135
rural_consumption_expenditures    208
urban_disposable_income            71
rural_disposable_income           144
dtype: int64


保存

In [51]:
living.to_csv(
    OUTPUT_DIR + "living_clean.csv",
    index=False
)